# common methods


In [1]:
# filter map and concatenate

import multiprocessing
print(multiprocessing.cpu_count())

8


In [6]:
import datasets

imdb_dataset=datasets.load_dataset("stanfordnlp/imdb") # we are not using the unsupervised split
_ = imdb_dataset.pop('unsupervised')
print(imdb_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})


# filtering

In [7]:
# Suppose we want to ensure that the number of words in a sentence should be of at least 100

print("Before filtering")
print(20*"-")
print(imdb_dataset)

Before filtering
--------------------
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})


In [8]:
num_words=100
imdb_filtered_dataset=imdb_dataset.filter(lambda example: len(example['text'].split())>=num_words)
print("After filtering")
print(20*"-")
print(imdb_filtered_dataset)

Filter: 100%|██████████| 25000/25000 [00:00<00:00, 70744.66 examples/s]

After filtering
--------------------
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 22074
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 21909
    })
})


# Map
- add prefix IMDB: to all the samples in the dataset

In [10]:
def add_prefix(example):
    example['text'] = "IMDB: " + example['text']
    return example


imdb_prefixed_dataset=imdb_filtered_dataset.map(add_prefix)
print("After adding prefix")
print(imdb_prefixed_dataset['train']['text'][1000])


After adding prefix
IMDB: This "film" is one of the most dreadful things I have ever seen.<br /><br />Please do yourselves a favor and avoid this incompetent concoction.<br /><br />Shaking the camera and having your actors adopt scowls does not count as "direction", which this film needed in droves. Not that the writing was all the wonderful, rather we were left with a bunch of completely artificial characters directed in that most artificial way (the pseudo-documentary "style" prized by those who don't know how to direct).<br /><br />This film gives the impression that it was done cynically to appeal to critics who don't know the first thing about film-making (which is most of them).<br /><br />Just terrible. It says a lot about Sundance and what it's become that Victor Vargas was showcased there.


## Concatenate Datasets

- Often we need to combine two or more datasets (say, Imdb review with rotten tomatoes) to create a larger dataset

- This is a typical approach for building pretraining datasets

- The only requirement is that the datasets must have same features and same number of splits

In [15]:
from datasets import load_dataset
from pprint import pprint
imdb_dataset_whole=load_dataset("stanfordnlp/imdb",split="train+test")
print(imdb_dataset_whole)
pprint(imdb_dataset_whole.features)

Dataset({
    features: ['text', 'label'],
    num_rows: 50000
})
{'label': ClassLabel(names=['neg', 'pos']), 'text': Value('string')}


In [16]:
rt_dataset_whole=load_dataset("cornell-movie-review-data/rotten_tomatoes",split="all")
print(rt_dataset_whole)
print(rt_dataset_whole.features)

Generating test split: 100%|██████████| 1066/1066 [00:00<00:00, 70582.64 examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 10662
})
{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}


In [17]:
concate_dataset=datasets.concatenate_datasets([imdb_dataset_whole,rt_dataset_whole])
print(concate_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 60662
})


## Interleaving Datasets
- Often we have n skewed datasets (that is number of samples in the dataset
might differ drastically)


- So we need to build a new dataset by intervaling the samples from each
dataset according to its distribution

- In the above case, IMDB has 50000 samples whereas rotten tomatoes has
10000 samples

- Let's build a dataset that contains 60% of samples from IMDB and 40% fror
rotten tomatoes

In [1]:
from datasets import interleave_datasets

inter_datasets=interleave_datasets([imdb_dataset_whole,rt_dataset_whole],probabilities=[0.6,0.4],seed=42)
print(inter_datasets)



d:\gokul_repos\iitm\dlp\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'imdb_dataset_whole' is not defined

- the original had 50k + 10k samples, so op should have been something else, here we take the min 
- the output has only 26k rows because, so once one dataset is exhauseted , we should stop the interleaving process.

26k => (60% of 10k) + (40% of 50k) => 6k+20k => 26k

- default , `stopping_strategy=first_exhausted`

## Iterable Dataset


In [20]:

imdb_iter_dataset=load_dataset("stanfordnlp/imdb",split="train",streaming=True)
print(imdb_iter_dataset)

IterableDataset({
    features: ['text', 'label'],
    num_shards: 1
})


In [22]:
for example in imdb_iter_dataset:
    pprint(example)
    break

{'label': 0,
 'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the '
         'controversy that surrounded it when it was first released in 1967. I '
         'also heard that at first it was seized by U.S. customs if it ever '
         'tried to enter this country, therefore being a fan of films '
         'considered "controversial" I really had to see this for myself.<br '
         '/><br />The plot is centered around a young Swedish drama student '
         'named Lena who wants to learn everything she can about life. In '
         'particular she wants to focus her attentions to making some sort of '
         'documentary on what the average Swede thought about certain '
         'political issues such as the Vietnam War and race issues in the '
         'United States. In between asking politicians and ordinary denizens '
         'of Stockholm about their opinions on politics, she has sex with her '
         'drama teacher, classmates, and married men.<br

-  the methods like `map,filter` works on the fly

In [23]:
for example in imdb_iter_dataset.map(add_prefix):
    pprint(example)
    break

{'label': 0,
 'text': 'IMDB: I rented I AM CURIOUS-YELLOW from my video store because of '
         'all the controversy that surrounded it when it was first released in '
         '1967. I also heard that at first it was seized by U.S. customs if it '
         'ever tried to enter this country, therefore being a fan of films '
         'considered "controversial" I really had to see this for myself.<br '
         '/><br />The plot is centered around a young Swedish drama student '
         'named Lena who wants to learn everything she can about life. In '
         'particular she wants to focus her attentions to making some sort of '
         'documentary on what the average Swede thought about certain '
         'political issues such as the Vietnam War and race issues in the '
         'United States. In between asking politicians and ordinary denizens '
         'of Stockholm about their opinions on politics, she has sex with her '
         'drama teacher, classmates, and married m

# QUESTIONS TO PONDER
- Can we load a dataset directly from external links?

- Can we load a dataset from zipped file directly?